# Finales Two-Stage-Modell für Human Activity Recognition

Finales Modell für Human Activity Recognition  
=============================================  
  
Ziel der Datei  
--------------  
Diese Datei enthält den vollständigen, dokumentierten finalen Modellablauf:  
  
1. Laden der Rohdaten `measures.csv` und `to_predict.csv`  
2. Erzeugen des offiziellen Train/Test-Splits nach Subject  
   - Testsubjects: 27, 28, 29, 30  
3. Bewertung des finalen Two-Stage-Modells auf dem offiziellen Testset  
4. Training des finalen Modells auf allen gelabelten Daten aus `measures.csv`  
5. Vorhersage der Aktivitäten für `to_predict.csv`  
6. Speichern der Datei `results/my_prediction.csv`  
  
Finales Modell  
--------------  
Das finale Modell ist ein zweistufiges Modell:  
  
Stufe 1: Hauptmodell für alle sechs Aktivitäten  
    Soft Voting Ensemble aus:  
    - Polynomial-SVM  
    - XGBoost  
    mit Gewichtung [1, 3], also XGBoost dreifach stärker gewichtet.  
  
Stufe 2: Spezialmodell für SITTING/STANDING  
    Falls das Hauptmodell SITTING oder STANDING vorhersagt, entscheidet ein  
    spezialisiertes Modell erneut nur zwischen diesen beiden Klassen.  
    Dieses Spezialmodell besteht aus:  
    - StandardScaler  
    - SelectKBest(k=150)  
    - Polynomial-SVM  
    - Threshold-Tuning mit threshold = -0.35  
  
Erwartete offizielle Testleistung  
---------------------------------  
Auf dem offiziellen Testset mit Subjects 27, 28, 29 und 30 wurde erreicht:  
  
    Test Accuracy: 0.976431  
    Test Error:    0.023569  
    Fehler:        35 von 1485  
  
Hinweis zur wissenschaftlichen Interpretation  
---------------------------------------------  
Das Two-Stage-Modell wurde aus der Fehleranalyse motiviert, da die meisten  
Fehler zwischen SITTING und STANDING lagen. Für die Prediction ist das Modell  
praktisch das beste getestete Modell. Im Bericht sollte erwähnt werden, dass  
Two-Stage und Threshold-Tuning explorative Optimierungsschritte waren.

## 0. Imports

In [2]:
# ============================================================
# 0. Imports
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

## 1. Globale Einstellungen und Pfade

In [3]:
# ============================================================
# 1. Globale Einstellungen und Pfade
# ============================================================

RANDOM_STATE = 42

RAW_DIR = Path("../../data/raw")
RESULTS_DIR = Path("../../results")
TABLE_DIR = RESULTS_DIR / "tables"
FIGURE_DIR = RESULTS_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MEASURES_PATH = RAW_DIR / "measures.csv"
TO_PREDICT_PATH = RAW_DIR / "to_predict.csv"
PREDICTION_PATH = RESULTS_DIR / "my_prediction.csv"

TARGET_COL = "activity"
GROUP_COL = "subject"
TEST_SUBJECTS = [27, 28, 29, 30]
BINARY_CLASSES = ["SITTING", "STANDING"]

# Schwellenwert aus dem besten getesteten Two-Stage-Modell.
SPECIAL_THRESHOLD = -0.35

## 2. Hilfsfunktionen

In [4]:
# ============================================================
# 2. Hilfsfunktionen
# ============================================================

def load_data():
    """Lädt die Rohdaten und gibt measures sowie to_predict zurück."""
    measures = pd.read_csv(MEASURES_PATH, sep=";", decimal=",")
    to_predict = pd.read_csv(TO_PREDICT_PATH, sep=";", decimal=",")

    print("Measures:", measures.shape)
    print("To predict:", to_predict.shape)

    return measures, to_predict



def get_feature_columns(measures):
    """Bestimmt alle Modellfeatures: alle Spalten außer subject und activity."""
    feature_cols = [
        col for col in measures.columns
        if col not in [TARGET_COL, GROUP_COL]
    ]

    print("Anzahl Features:", len(feature_cols))
    return feature_cols



def make_official_split(measures):
    """
    Erzeugt den offiziellen Train/Test-Split nach Subject.

    Testset:
        Subjects 27, 28, 29, 30
    Trainingsset:
        alle übrigen Subjects
    """
    train_data = measures[~measures[GROUP_COL].isin(TEST_SUBJECTS)].copy()
    test_data = measures[measures[GROUP_COL].isin(TEST_SUBJECTS)].copy()

    print("\nOfficial split")
    print("Train data:", train_data.shape)
    print("Test data:", test_data.shape)
    print("Train subjects:", sorted(train_data[GROUP_COL].unique()))
    print("Test subjects:", sorted(test_data[GROUP_COL].unique()))

    overlap = set(train_data[GROUP_COL]).intersection(set(test_data[GROUP_COL]))
    print("Subject overlap:", overlap)

    if len(overlap) != 0:
        raise ValueError("Train/Test-Split enthält überlappende Subjects.")

    return train_data, test_data



def build_main_model(label_encoder, use_gpu=True):
    """
    Baut das Hauptmodell für alle sechs Klassen.

    Hauptmodell:
        Soft Voting aus Polynomial-SVM und XGBoost.
        Gewichtung [1, 3] bedeutet:
        - Polynomial-SVM Gewicht 1
        - XGBoost Gewicht 3
    """
    weight_by_label = {
        "LAYING": 1.0,
        "SITTING": 2.0,
        "STANDING": 1.5,
        "WALKING": 1.2,
        "WALKING_DOWNSTAIRS": 3.0,
        "WALKING_UPSTAIRS": 2.0,
    }

    class_weights_encoded = {
        encoded: weight_by_label[label]
        for encoded, label in enumerate(label_encoder.classes_)
    }

    poly_svm_main = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("classifier", SVC(
            kernel="poly",
            C=1e5,
            tol=1e-5,
            shrinking=True,
            gamma=0.001,
            degree=6,
            coef0=2.0,
            class_weight=class_weights_encoded,
            probability=True,
            random_state=RANDOM_STATE,
        )),
    ])

    xgb_params = dict(
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        n_estimators=300,
        max_depth=2,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=1.0,
        min_child_weight=3,
        reg_lambda=5,
        reg_alpha=0.1,
    )

    if use_gpu:
        xgb_params["device"] = "cuda"

    xgb_main = XGBClassifier(**xgb_params)

    main_model = VotingClassifier(
        estimators=[
            ("poly_svm", poly_svm_main),
            ("xgb", xgb_main),
        ],
        voting="soft",
        weights=[1, 3],
        n_jobs=-1,
    )

    return main_model



def build_special_model():
    """
    Baut das Spezialmodell für SITTING vs. STANDING.

    Spezialmodell:
        - StandardScaler
        - SelectKBest mit k=150
        - Polynomial-SVM

    Die Parameter entsprechen dem besten getesteten Spezialmodell.
    """
    special_model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("select", SelectKBest(score_func=f_classif, k=150)),
        ("classifier", SVC(
            kernel="poly",
            C=10,
            gamma=0.0003,
            degree=3,
            coef0=2.0,
            class_weight={"SITTING": 1.4, "STANDING": 1.0},
            random_state=RANDOM_STATE,
        )),
    ])

    return special_model



def two_stage_predict(main_model, special_model, label_encoder, X, threshold=SPECIAL_THRESHOLD):
    """
    Erzeugt Two-Stage-Vorhersagen.

    1. Hauptmodell sagt alle sechs Klassen vorher.
    2. Falls Hauptmodell SITTING oder STANDING vorhersagt, wird die Beobachtung
       durch das Spezialmodell erneut zwischen SITTING und STANDING bewertet.
    3. Die Entscheidung des Spezialmodells wird über die decision_function und
       den festen Threshold getroffen.
    """
    main_pred_encoded = main_model.predict(X)
    main_pred = label_encoder.inverse_transform(main_pred_encoded)

    final_pred = main_pred.copy()

    mask_sitting_standing = pd.Series(main_pred).isin(BINARY_CLASSES).values

    if mask_sitting_standing.sum() > 0:
        scores = special_model.decision_function(X[mask_sitting_standing])
        classes = special_model.named_steps["classifier"].classes_

        negative_class = classes[0]
        positive_class = classes[1]

        special_pred = np.where(
            scores >= threshold,
            positive_class,
            negative_class,
        )

        final_pred[mask_sitting_standing] = special_pred

    return final_pred, main_pred



def fit_two_stage_model(train_data, feature_cols, use_gpu=True):
    """
    Trainiert Hauptmodell und Spezialmodell auf den übergebenen Trainingsdaten.
    """
    X_train = train_data[feature_cols]
    y_train = train_data[TARGET_COL]

    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)

    main_model = build_main_model(label_encoder, use_gpu=use_gpu)

    try:
        main_model.fit(X_train, y_train_encoded)
    except Exception as error:
        if use_gpu:
            print("\nGPU-Version von XGBoost hat nicht funktioniert.")
            print("Fehler:", error)
            print("Starte Hauptmodell erneut ohne GPU...")
            main_model = build_main_model(label_encoder, use_gpu=False)
            main_model.fit(X_train, y_train_encoded)
        else:
            raise error

    binary_train = train_data[train_data[TARGET_COL].isin(BINARY_CLASSES)].copy()

    X_train_binary = binary_train[feature_cols]
    y_train_binary = binary_train[TARGET_COL]

    special_model = build_special_model()
    special_model.fit(X_train_binary, y_train_binary)

    return main_model, special_model, label_encoder



def evaluate_on_official_test(train_data, test_data, feature_cols):
    """
    Trainiert das finale Modell auf dem offiziellen Trainingsset und bewertet
    es auf dem offiziellen Testset.
    """
    print("\n============================================================")
    print("Offizielle Testset-Bewertung")
    print("============================================================")

    main_model, special_model, label_encoder = fit_two_stage_model(
        train_data=train_data,
        feature_cols=feature_cols,
        use_gpu=True,
    )

    X_test = test_data[feature_cols]
    y_test = test_data[TARGET_COL]

    final_pred, main_pred = two_stage_predict(
        main_model=main_model,
        special_model=special_model,
        label_encoder=label_encoder,
        X=X_test,
        threshold=SPECIAL_THRESHOLD,
    )

    test_accuracy = accuracy_score(y_test, final_pred)
    test_error = 1 - test_accuracy
    n_wrong = int((y_test.values != final_pred).sum())
    n_correct = int((y_test.values == final_pred).sum())

    print("Test Accuracy:", round(test_accuracy, 6))
    print("Test Error:", round(test_error, 6))
    print("Richtige Vorhersagen:", n_correct)
    print("Falsche Vorhersagen:", n_wrong)

    # Classification Report speichern
    report_df = pd.DataFrame(
        classification_report(y_test, final_pred, output_dict=True)
    ).transpose()

    report_df.to_csv(
        TABLE_DIR / "final_two_stage_test_classification_report.csv",
        index=True,
    )

    # Confusion Matrix speichern
    labels = list(label_encoder.classes_)

    cm = confusion_matrix(y_test, final_pred, labels=labels)
    cm_table = pd.DataFrame(
        cm,
        index=[f"True: {label}" for label in labels],
        columns=[f"Pred: {label}" for label in labels],
    )

    cm_table.to_csv(
        TABLE_DIR / "final_two_stage_test_confusion_matrix.csv",
        index=True,
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=labels,
    )

    fig, ax = plt.subplots(figsize=(9, 7))
    disp.plot(ax=ax, xticks_rotation=45, values_format="d")
    plt.title("Confusion Matrix for Final Two-Stage Model")
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "final_two_stage_test_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    # Fehleranalyse speichern
    error_df = test_data.copy()
    error_df["prediction"] = final_pred
    error_df["correct"] = error_df[TARGET_COL] == error_df["prediction"]
    wrong_df = error_df[~error_df["correct"]].copy()

    wrong_df[[GROUP_COL, TARGET_COL, "prediction", "correct"]].to_csv(
        TABLE_DIR / "final_two_stage_test_errors.csv",
        index=False,
    )

    print("\nFehler pro echter Klasse:")
    print(wrong_df[TARGET_COL].value_counts())

    print("\nFehler nach echter Klasse und Vorhersage:")
    print(wrong_df[[TARGET_COL, "prediction"]].value_counts())

    print("\nFehler pro Subject:")
    print(wrong_df[GROUP_COL].value_counts().sort_index())

    metrics_df = pd.DataFrame({
        "model": ["Final_TwoStage_SoftVoting_PolySVM_XGBoost_plus_PolySVM_Special"],
        "test_accuracy": [test_accuracy],
        "test_error": [test_error],
        "n_test_observations": [len(y_test)],
        "n_correct": [n_correct],
        "n_wrong": [n_wrong],
        "threshold": [SPECIAL_THRESHOLD],
    })

    metrics_df.to_csv(
        TABLE_DIR / "final_two_stage_test_metrics.csv",
        index=False,
    )

    return metrics_df



def create_final_prediction(measures, to_predict, feature_cols):
    """
    Trainiert das finale Two-Stage-Modell auf allen gelabelten Daten aus
    measures.csv und erzeugt die finale Datei my_prediction.csv.
    """
    print("\n============================================================")
    print("Finales Modell auf allen gelabelten Daten trainieren")
    print("============================================================")

    main_model, special_model, label_encoder = fit_two_stage_model(
        train_data=measures,
        feature_cols=feature_cols,
        use_gpu=True,
    )

    X_unseen = to_predict[feature_cols]

    predictions, main_predictions = two_stage_predict(
        main_model=main_model,
        special_model=special_model,
        label_encoder=label_encoder,
        X=X_unseen,
        threshold=SPECIAL_THRESHOLD,
    )

    # Für die Challenge: eine Spalte, kein Header, kein Index.
    pd.Series(predictions).to_csv(
        PREDICTION_PATH,
        index=False,
        header=False,
    )

    print("Finale Vorhersagen gespeichert:", PREDICTION_PATH)
    print("Anzahl Vorhersagen:", len(predictions))

    print("\nVorhersageverteilung:")
    print(pd.Series(predictions).value_counts())

    # Zusatzdatei mit subject und Prediction zur Kontrolle.
    prediction_check_df = to_predict[[GROUP_COL]].copy()
    prediction_check_df["main_prediction"] = main_predictions
    prediction_check_df["final_prediction"] = predictions

    prediction_check_df.to_csv(
        TABLE_DIR / "final_prediction_with_subjects_for_check.csv",
        index=False,
    )

    return predictions

## 3. Main Execution

In [5]:
# ============================================================
# 3. Main Execution
# ============================================================

if __name__ == "__main__":
    measures, to_predict = load_data()
    feature_cols = get_feature_columns(measures)

    # Kurze Datenqualitätsprüfung
    print("\nDatenqualität")
    print("Missing values in measures:", measures.isna().sum().sum())
    print("Missing values in to_predict:", to_predict.isna().sum().sum())
    print("Duplicate rows in measures:", measures.duplicated().sum())
    print("Duplicate rows in to_predict:", to_predict.duplicated().sum())

    # Offizielle Testset-Bewertung
    train_data, test_data = make_official_split(measures)
    evaluate_on_official_test(train_data, test_data, feature_cols)

    # Finale Prediction für to_predict.csv
    create_final_prediction(measures, to_predict, feature_cols)

    print("\nFertig.")

Measures: (7352, 563)
To predict: (2947, 562)
Anzahl Features: 561

Datenqualität
Missing values in measures: 0
Missing values in to_predict: 0
Duplicate rows in measures: 0
Duplicate rows in to_predict: 0

Official split
Train data: (5867, 563)
Test data: (1485, 563)
Train subjects: [np.float64(1.0), np.float64(3.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(11.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(19.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(25.0), np.float64(26.0)]
Test subjects: [np.float64(27.0), np.float64(28.0), np.float64(29.0), np.float64(30.0)]
Subject overlap: set()

Offizielle Testset-Bewertung


C:\Users\mluxe\AppData\Roaming\Python\Python313\site-packages\xgboost\core.py:751: UserWarning: [08:18:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Test Accuracy: 0.976431
Test Error: 0.023569
Richtige Vorhersagen: 1450
Falsche Vorhersagen: 35

Fehler pro echter Klasse:
activity
SITTING     29
STANDING     6
Name: count, dtype: int64

Fehler nach echter Klasse und Vorhersage:
activity  prediction
SITTING   STANDING      29
STANDING  SITTING        6
Name: count, dtype: int64

Fehler pro Subject:
subject
27.0     1
28.0    27
29.0     5
30.0     2
Name: count, dtype: int64

Finales Modell auf allen gelabelten Daten trainieren
Finale Vorhersagen gespeichert: ..\..\results\my_prediction.csv
Anzahl Vorhersagen: 2947

Vorhersageverteilung:
STANDING              593
LAYING                537
WALKING               524
WALKING_UPSTAIRS      467
SITTING               428
WALKING_DOWNSTAIRS    398
Name: count, dtype: int64

Fertig.
